In [37]:
import pandas as pd
import matplotlib.pyplot as plt
import os

In [38]:
# --- Configuração das variantes ---
variants = {
    "Baseline": {
        "path": r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Baseline\statistical_summary.csv",
        "color": "blue",
        "x_col": "Generation",
    },
    "Crossover Variant (Two-Point)": {
        "path": r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Crossover_Variant_TwoPoint\statistical_summary.csv",
        "color": "green",
        "x_col": "Generation",
    },
    "Mutation Variant (Inversion)": {
        "path": r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Mutation_Variant_Inversion\statistical_summary.csv",
        "color": "red",
        "x_col": "Generation",
    },
}

output_dir = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Plots"
os.makedirs(output_dir, exist_ok=True)

In [39]:
def read_csv_clean(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()  # remove espaços e BOM invisíveis
    return df

In [40]:
def normalize_x(df, x_col):
    """Normaliza o eixo X para percentagem de progresso (0–100%)."""
    x = df[x_col]
    return (x - x.min()) / (x.max() - x.min()) * 100

In [41]:
def plot_variants(metric_col, ylabel, title, filename, show_std=False, normalize=True):
    fig, ax = plt.subplots(figsize=(12, 6))
    plotted = False

    for label, cfg in variants.items():
        path = cfg["path"]
        color = cfg["color"]
        x_col = cfg["x_col"]

        if not os.path.exists(path):
            print(f"File not Found: {path}")
            continue

        df = read_csv_clean(path)
        # Debug: mostra colunas se x_col não for encontrado
        if x_col not in df.columns:
            print(f"[ERRO] Coluna '{x_col}' não encontrada em '{path}'.")
            print(f"       Colunas disponíveis: {list(df.columns)}")
            continue

        x = normalize_x(df, x_col) if normalize else df[x_col]

        ax.plot(x, df[metric_col],
                label=label, color=color, linewidth=2)

        if show_std and "Std_Dev" in df.columns:
            ax.fill_between(
                x,
                df[metric_col] - df["Std_Dev"],
                df[metric_col] + df["Std_Dev"],
                color=color, alpha=0.15
            )
        plotted = True

    if not plotted:
        print(f"No file found for '{title}'.")
        plt.close()
        return

    xlabel = "evolution (%)\n(GA: Generations  |  SA: Iterations)" if normalize else "Steps"
    ax.set_title(title, fontsize=14)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.legend(loc="upper right", fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.6)
    fig.tight_layout()

    out_path = os.path.join(output_dir, filename)
    fig.savefig(out_path, dpi=300)
    plt.close()
    print(f"Plot saved in : {out_path}")

In [42]:
plot_variants(
    metric_col="Mean_RMSE",
    ylabel="Mean RMSE",
    title="Variant Comparison (Mean RMSE)",
    filename="mean_variant_comparison.png",
    show_std=True,
    normalize=True
)

plot_variants(
    metric_col="Best_RMSE",
    ylabel="RMSE",
    title="Variant Comparison (Best Solution)",
    filename="best_variant_comparison.png",
    show_std=False,
    normalize=True
)

Plot saved in : C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Plots\mean_variant_comparison.png
Plot saved in : C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Plots\best_variant_comparison.png


In [43]:
variants = {
            "Simulated Annealing": {
        "path": r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Simulated_Annealing\statistical_summary.csv",
        "color": "red",
        "x_col": "Iteration"
    },
            "Best Variant (Mutation: Inversion)": {
        "path": r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Mutation_Variant_Inversion\statistical_summary.csv",
        "color": "blue",
        "x_col": "Generation",
    },
}

In [44]:
plot_variants(
    metric_col="Mean_RMSE",
    ylabel="Mean RMSE",
    title="Challenge 2: GA Vs SA (Mean RMSE)",
    filename="mean_c2_comparison.png",
    show_std=True
)

plot_variants(
    metric_col="Best_RMSE",
    ylabel="RMSE",
    title="Challenge 2: GA Vs SA (Best Solution)",
    filename="best_c2_comparison.png",
    show_std=False
)

Plot saved in : C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Plots\mean_c2_comparison.png
Plot saved in : C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Plots\best_c2_comparison.png


## Crossover Vs Mutation Variant

In [45]:
from scipy import stats
import pandas as pd

path_mutation_variant  = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Mutation_Variant_Inversion\raw_fitness_data_50.csv"
path_crossover_variant = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Crossover_Variant_TwoPoint\raw_fitness_data_50.csv"

mutation_variant_data  = pd.read_csv(path_mutation_variant,  index_col=0)
crossover_variant_data = pd.read_csv(path_crossover_variant, index_col=0)

mutation_variant_final  = mutation_variant_data.iloc[:, -1].astype(float)
crossover_variant_final = crossover_variant_data.iloc[:, -1].astype(float)

print(f"Mutation Variant final RMSE: mean={mutation_variant_final.mean():.4f}  std={mutation_variant_final.std():.4f}")
print(f"Crossover Variant final RMSE: mean={crossover_variant_final.mean():.4f}  std={crossover_variant_final.std():.4f}")

# Mann-Whitney U
u_stat, p_val = stats.mannwhitneyu(mutation_variant_final, crossover_variant_final, alternative='less')

print(f"\nStatistical Test: Mann-Whitney U")
print(f"U-statistic: {u_stat:.4f}")
print(f"p-value:     {p_val:.5f}")
if p_val < 0.05:
    print("There is STATISTICALLY SIGNIFICANT evidence that Mutation variant performs better than Crossover variant (Mutation < Crossover).")
else:
    print("No statistical evidence of difference between Mutation and Crossover variants.")

# Success Rate
limiar = 30.0
sr_mutation  = (mutation_variant_final  < limiar).mean()
sr_crossover = (crossover_variant_final < limiar).mean()

print(f"\nSuccess Rate (RMSE < {limiar})")
print(f"Mutation Variant:  {sr_mutation:.2%}")
print(f"Crossover Variant: {sr_crossover:.2%}")

Mutation Variant final RMSE: mean=30.4000  std=1.6746
Crossover Variant final RMSE: mean=35.7950  std=1.8973

Statistical Test: Mann-Whitney U
U-statistic: 9.0000
p-value:     0.00000
There is STATISTICALLY SIGNIFICANT evidence that Mutation variant performs better than Crossover variant (Mutation < Crossover).

Success Rate (RMSE < 30.0)
Mutation Variant:  55.00%
Crossover Variant: 0.00%


## Baseline Vs Best Variant (Mutation_Variant_TwoPoint)

In [46]:
from scipy import stats
import pandas as pd

path_baseline     = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Baseline\raw_fitness_data_50.csv"
path_best_variant = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Mutation_Variant_Inversion\raw_fitness_data_50.csv"

baseline_data     = pd.read_csv(path_baseline,     index_col=0)
best_variant_data = pd.read_csv(path_best_variant, index_col=0)  # ← era path_best_variant.iloc

baseline_final     = baseline_data.iloc[:, -1].astype(float)
best_variant_final = best_variant_data.iloc[:, -1].astype(float)  # ← corrigido

print(f"Baseline     final RMSE: mean={baseline_final.mean():.4f}  std={baseline_final.std():.4f}")
print(f"Best Variant final RMSE: mean={best_variant_final.mean():.4f}  std={best_variant_final.std():.4f}")

# Mann-Whitney U
u_stat, p_val = stats.mannwhitneyu(baseline_final, best_variant_final, alternative='less')

print(f"\nStatistical Test: Mann-Whitney U")
print(f"U-statistic: {u_stat:.4f}")
print(f"p-value:     {p_val:.5f}")
if p_val < 0.05:
    print("There is STATISTICALLY SIGNIFICANT evidence that Mutation variant performs better than Crossover variant (Mutation < Crossover).")
else:
    print("No statistical evidence of difference between Mutation and Crossover variants.")

# Success Rate
limiar = 30.0
sr_baseline     = (baseline_final < limiar).mean()
sr_best_variant = (best_variant_final < limiar).mean()

print(f"\nSuccess Rate (RMSE < {limiar})")
print(f"Baseline:     {sr_baseline:.2%}")
print(f"Best Variant: {sr_best_variant:.2%}")

Baseline     final RMSE: mean=30.4668  std=1.3379
Best Variant final RMSE: mean=30.4000  std=1.6746

Statistical Test: Mann-Whitney U
U-statistic: 214.0000
p-value:     0.65257
No statistical evidence of difference between Mutation and Crossover variants.

Success Rate (RMSE < 30.0)
Baseline:     40.00%
Best Variant: 55.00%


## Genetic Algorithms Vs Simulaetd Annealing

In [48]:
from scipy import stats
import pandas as pd

path_ga = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Mutation_Variant_Inversion\raw_fitness_data_50.csv"
path_sa = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Simulated_Annealing\raw_fitness_data_50.csv"

ga_data = pd.read_csv(path_ga, index_col=0)
sa_data = pd.read_csv(path_sa, index_col=0)

ga_final = ga_data.iloc[:, -1].astype(float)
sa_final = sa_data.iloc[:, -1].astype(float)

print(f"Genetic Algorithm (Mutation Inversion) final RMSE: mean={ga_final.mean():.4f}  std={ga_final.std():.4f}")
print(f"Simulated Annealing final RMSE: mean={sa_final.mean():.4f}  std={sa_final.std():.4f}")

# Mann-Whitney U — alternative='less' testa se GA produz valores MENORES (melhores) que SA
u_stat, p_val = stats.mannwhitneyu(ga_final, sa_final, alternative='less')

print(f"\nStatistical Test: Mann-Whitney U")
print(f"U-statistic: {u_stat:.4f}")
print(f"p-value:     {p_val:.5f}")
if p_val < 0.05:
    print("There is STATISTICALLY SIGNIFICANT evidence that GA performs better than SA (GA RMSE < SA RMSE).")
else:
    print("No statistical evidence that GA outperforms SA.")

# Success Rate
limiar = 30.0
sr_ga = (ga_final < limiar).mean()
sr_sa = (sa_final < limiar).mean()

print(f"\nSuccess Rate (RMSE < {limiar})")
print(f"GA: {sr_ga:.2%}")
print(f"SA: {sr_sa:.2%}")

Genetic Algorithm (Mutation Inversion) final RMSE: mean=30.4000  std=1.6746
Simulated Annealing final RMSE: mean=35.1410  std=4.0949

Statistical Test: Mann-Whitney U
U-statistic: 47.0000
p-value:     0.00002
There is STATISTICALLY SIGNIFICANT evidence that GA performs better than SA (GA RMSE < SA RMSE).

Success Rate (RMSE < 30.0)
GA: 55.00%
SA: 5.00%


In [2]:
from scipy import stats
import pandas as pd

path_ga = r"..\Project\data\results\Baseline\raw_fitness_data_50.csv"
path_sa = r"C:..\Project\data\results\Simulated_Annealing\raw_fitness_data_50.csv"

ga_data = pd.read_csv(path_ga, index_col=0)
sa_data = pd.read_csv(path_sa, index_col=0)

ga_final = ga_data.iloc[:, -1].astype(float)
sa_final = sa_data.iloc[:, -1].astype(float)

print(f"Genetic Algorithm (Baseline) final RMSE: mean={ga_final.mean():.4f}  std={ga_final.std():.4f}")
print(f"Simulated Annealing final RMSE: mean={sa_final.mean():.4f}  std={sa_final.std():.4f}")

# Mann-Whitney U — alternative='less' testa se GA produz valores MENORES (melhores) que SA
u_stat, p_val = stats.mannwhitneyu(ga_final, sa_final, alternative='less')

print(f"\nStatistical Test: Mann-Whitney U")
print(f"U-statistic: {u_stat:.4f}")
print(f"p-value:     {p_val:.5f}")
if p_val < 0.05:
    print("There is STATISTICALLY SIGNIFICANT evidence that GA performs better than SA (GA RMSE < SA RMSE).")
else:
    print("No statistical evidence that GA outperforms SA.")

# Success Rate
limiar = 30.0
sr_ga = (ga_final < limiar).mean()
sr_sa = (sa_final < limiar).mean()

print(f"\nSuccess Rate (RMSE < {limiar})")
print(f"GA: {sr_ga:.2%}")
print(f"SA: {sr_sa:.2%}")

Genetic Algorithm (Baseline) final RMSE: mean=30.4668  std=1.3379
Simulated Annealing final RMSE: mean=35.1410  std=4.0949

Statistical Test: Mann-Whitney U
U-statistic: 54.0000
p-value:     0.00004
There is STATISTICALLY SIGNIFICANT evidence that GA performs better than SA (GA RMSE < SA RMSE).

Success Rate (RMSE < 30.0)
GA: 40.00%
SA: 5.00%
